# Precompute features on Kaggle

This notebook prepares the Kaggle runtime to run the project's preprocessing pipeline (`scripts/precompute_features.py`) and writes a per-CSV feature cache that can be reused for model training.

Usage notes:
- Upload the repository files to the Kaggle notebook (or set `REPO_URL` to clone the repo).
- Place your dataset (the folder that contains `Router/` and `router no person/`) in the Kaggle input or adjust `DATASET_DIR` below.
- The notebook writes cache files to `/kaggle/working/feature_cache` by default.

In [ ]:
# Configuration: set REPO_URL if you want to clone the repo automatically, otherwise upload files to the notebook
REPO_URL = ''  # e.g. 'https://github.com/yourname/WiVSD.git'
# Kaggle mounts uploaded datasets under /kaggle/input. If you uploaded the dataset as a dataset, set DATASET_SLUG accordingly, otherwise leave it None and upload repo+data to working dir.
DATASET_SLUG = None  # set to folder name under /kaggle/input if applicable
# Where to store the precomputed feature cache inside the Kaggle working dir
FEATURE_CACHE_DIR = '/kaggle/working/feature_cache'
# How many CPU workers to use for preprocessing
N_JOBS = 2

print('CONFIG:')
print(' REPO_URL =', REPO_URL)
print(' DATASET_SLUG =', DATASET_SLUG)
print(' FEATURE_CACHE_DIR =', FEATURE_CACHE_DIR)
print(' N_JOBS =', N_JOBS)

In [ ]:
# Install or upgrade minimal dependencies (scikit-learn, joblib, tqdm). Kaggle already includes many packages, but this ensures recent versions.
!pip install -q --upgrade scikit-learn joblib tqdm

In [ ]:
import os
from pathlib import Path
# If repository not uploaded, optionally clone it
if REPO_URL and not Path('src').exists():
    print('Cloning repo...')
    os.system(f'git clone {REPO_URL} repo || true')
    # if cloned into repo, move files into working dir root (optional)
    if Path('repo').exists():
        os.system('rsync -a repo/ .')

print('Listing /kaggle/input (if any):')
os.system('ls -la /kaggle/input || true')

# Set ROOT: either dataset slug path under /kaggle/input or current working dir if you uploaded the data here
if DATASET_SLUG:
    ROOT = Path('/kaggle/input') / DATASET_SLUG
else:
    ROOT = Path('.')

print('Using ROOT =', ROOT)
print('Scripts directory contents:')
os.system('ls -la scripts || true')

If your dataset is inside `/kaggle/input/<slug>` and contains the subfolders `Router/` and `router no person/`, the notebook will use that as the root. Otherwise, upload the CSV folders into the notebook working directory before running the next cell.

In [ ]:
# Quick check for expected subfolders under ROOT
from pathlib import Path
root = Path('/kaggle/input') / DATASET_SLUG if DATASET_SLUG else Path('.')
print('ROOT exists:', root.exists())
print('Router exists:', (root / 'Router').exists())
print(
print('
Listing a few CSVs (if present):')
os.system(f'ls -la 
 2>/dev/null | sed -n 
1
10
 || true')
os.system(f'ls -la 
 2>/dev/null | sed -n 
1
10
 || true')
: 
,
: { 
: 
 },
: [
,
,
,
,
,
,
,
{root}
{feature_cache}
router no person
,

In [ ]:
# Show produced cache files and a small sample of one entry
from pathlib import Path
import numpy as np
fc = Path(FEATURE_CACHE_DIR)
print('Cache dir exists:', fc.exists())
files = sorted(fc.glob('*.npz'))
print('Found', len(files), 'npz files')
for f in files[:5]:
    print('-', f)
    try:
        d = np.load(f, allow_pickle=True)
        print('   features.shape=', d['features'].shape, ' label=', int(d['label']))
    except Exception as e:
        print('   failed to read', f, e)

Next steps:
- Use `scripts/train_ml_classifier.py` with `--feature-cache /kaggle/working/feature_cache` to train using the cached features.
- Download the feature cache (`/kaggle/working/feature_cache`) if you want to train locally or elsewhere.